# hftbacktest: the one line that sets a realistic queue model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarptandoven/queuetruth/blob/main/examples/notebooks/06_hftbacktest.ipynb)

hftbacktest's `BacktestAsset` needs a queue model and an exchange model, and the usual choice is a guess. This
notebook gets the setting two ways, the atlas when you have no order-level data and a fit when you do, applies it
with one line, and then checks an hftbacktest run order by order:

```python
asset = qthbt.configure(asset, qt.fit("hftbacktest", data="my-dataset"))
asset = qthbt.configure(asset, qt.atlas.calibration("hftbacktest", tier="T1"))
```

Data: queuetruth's **synthetic** dataset and the repository's **synthetic** atlas (invented numbers). The numbers
only show the workflow; they say nothing about a real venue. Every result reads the same way: `.summary` (the
answer), `.table()`, `.to_html()`, `.save(path)`, and a compact view in a notebook (`display(result)`).

In [1]:
# Setup: on Colab (or anywhere queuetruth is missing) this installs it from PyPI; elsewhere it does nothing.
import importlib
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("queuetruth") is None:
    try:
        from IPython import get_ipython

        shell = get_ipython()
    except ImportError:
        shell = None
    if shell is not None:  # %pip lets Colab offer a restart when an already-imported package was upgraded
        shell.run_line_magic("pip", "install -q queuetruth pandas matplotlib")
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "queuetruth", "pandas", "matplotlib"])
    importlib.invalidate_caches()
if importlib.util.find_spec("queuetruth") is None or importlib.util.find_spec("queuetruth.frames") is None:
    raise RuntimeError(
        "this notebook needs queuetruth.frames (first released after queuetruth 0.1.1): pip install -U queuetruth"
    )

In [2]:
import importlib.util
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

import queuetruth as qt

pd.set_option("display.width", 140)
WORK = Path("qt-notebooks") / "06_hftbacktest"
WORK.mkdir(parents=True, exist_ok=True)

# queuetruth's synthetic order-level dataset: two futures (QTAU6, QTBU6), two days, invented prices.
DATA = WORK / "synth"
if not (DATA / "manifest.json").exists():
    qt.fixtures_synth(str(DATA))

# The repository's synthetic atlas (invented numbers, no market data): the checkout's copy, else the one on GitHub.
# Leave `source=` out to use the published atlas (it asks before downloading anything).
_local = Path(qt.__file__).resolve().parents[2] / "tests" / "fixtures" / "atlas_synth"
ATLAS = (
    str(_local)
    if (_local / "atlas" / "v1" / "manifest.json").exists()
    else "https://raw.githubusercontent.com/sarptandoven/queuetruth/main/tests/fixtures/atlas_synth/atlas/v1/"
)

# Twelve resting limit orders on the synthetic futures, as a bar backtest reports them: placed at `placed` (UTC),
# and filled 5 s later when `filled` (the touch rule), else cancelled 30 s later.
ORDERS = pd.DataFrame(
    [
        ("O-1", "QTBU6", "sell", 1, 5008.50, "2026-06-16 14:38:42", True),
        ("O-2", "QTAU6", "buy", 2, 4999.75, "2026-06-16 14:38:05", False),
        ("O-3", "QTBU6", "sell", 3, 5010.25, "2026-06-15 17:09:27", True),
        ("O-4", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:32:15", False),
        ("O-5", "QTAU6", "sell", 2, 5002.75, "2026-06-15 15:39:13", True),
        ("O-6", "QTAU6", "buy", 3, 5004.00, "2026-06-15 19:04:05", False),
        ("O-7", "QTBU6", "buy", 1, 5008.75, "2026-06-15 15:36:19", True),
        ("O-8", "QTBU6", "sell", 2, 5008.50, "2026-06-16 16:58:59", False),
        ("O-9", "QTAU6", "buy", 3, 4999.50, "2026-06-16 14:39:20", True),
        ("O-10", "QTAU6", "buy", 1, 4999.50, "2026-06-16 17:25:05", False),
        ("O-11", "QTBU6", "buy", 2, 5008.25, "2026-06-16 16:12:29", True),
        ("O-12", "QTAU6", "buy", 3, 5001.25, "2026-06-15 15:05:05", False),
    ],
    columns=["order", "symbol", "side", "qty", "price", "placed", "filled"],
)
ORDERS["placed"] = pd.to_datetime(ORDERS["placed"], utc=True)
ORDERS["done"] = ORDERS["placed"] + pd.to_timedelta(ORDERS["filled"].map({True: 5, False: 30}), unit="s")


def have(*modules, hint=None):
    """True when every module is importable; else prints which are missing and how to get them."""
    missing = [m for m in modules if importlib.util.find_spec(m) is None]
    if missing:
        print(f"skipped: {', '.join(missing)} not installed here ({hint or 'pip install ' + ' '.join(missing)})")
    return not missing


print("queuetruth", qt.__version__, "| SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only")

queuetruth 0.3.0 | SYNTHETIC data and a SYNTHETIC atlas: the numbers show the workflow only


## 1. No data yet: the atlas's closest setting

`qt.atlas.calibration` reads the atlas's closest hftbacktest setting for a liquidity tier, as the same kind of
result `qt.fit` returns. Without `source=` it reads the published atlas, after asking.

In [3]:
cal = qt.atlas.calibration("hftbacktest", tier="ALL", source=ATLAS, yes=True)
display(cal)

setting,tier,horizon,bias_pp,ci_lo_pp,ci_hi_pp,verdict,days,model_id
hftbacktest Power3 queue model (n=3),ALL,1min,0.2,-0.1,0.5,no_clear_difference,3,hbt_power3[exch=partial;mode=faithful;n=3]


In [4]:
display(cal.settings)

{'queue_model': 'power_prob_queue_model3',
 'queue_model_args': [3.0],
 'exchange': 'no_partial_fill_exchange'}

## 2. Your data: fit the `power_prob_queue_model3` exponent

`qt.fit` fits `n` on the first 70% of the data and compares it with the library default `n = 2` on the rest. A
fit that does not beat the default there is refused, says why, and writes no snippet: here it is refused.

In [5]:
fit = qt.fit(backtester="hftbacktest", data=DATA, split="70%", out=WORK / "fit")
display(fit)

horizon,fitted_bias_pp,default_bias_pp,fitted_logloss,default_logloss,result
100ms,0,0,1e-06,1e-06,tie
1s,0,0,1e-06,1e-06,tie
10s,0.0159147,0.0159147,1e-06,1e-06,tie
1min,-0.634537,-0.634537,0.148236,0.088942,worse
5min,-0.506443,-1.42177,0.126459,0.189688,better


In [6]:
print("passed:", fit.passed)
display(fit.table())

passed: False


,horizon,fitted_bias_pp,default_bias_pp,fitted_logloss,default_logloss,result
0,100ms,0.000000,0.000000,0.000001,0.000001,tie
1,1s,0.000000,0.000000,0.000001,0.000001,tie
2,10s,0.015915,0.015915,0.000001,0.000001,tie
3,1min,-0.634537,-0.634537,0.148236,0.088942,worse
4,5min,-0.506443,-1.421775,0.126459,0.189688,better


## 3. The one line

`queuetruth_hftbacktest.configure` sets the result's queue model and exchange on your `BacktestAsset` and leaves
your data, tick, lot, latency and fee settings alone. A refused fit raises `CalibrationRefused`, so the line here
takes the atlas setting. `configure` uses the standard library only: in a separate hftbacktest venv, save the
result here (`cal.save("atlas-fit.json")`) and pass the path there.

In [7]:
cal.save(WORK / "atlas-fit.json")
if have("hftbacktest", "queuetruth_hftbacktest"):
    import queuetruth_hftbacktest as qthbt
    from hftbacktest import BacktestAsset

    asset = BacktestAsset().linear_asset(1.0).tick_size(0.25).lot_size(1.0)  # your settings
    asset = qthbt.configure(asset, cal)
    print("configured:", qthbt.settings(str(WORK / "atlas-fit.json")))
else:
    print("the settings the line applies:", cal.settings)

skipped: hftbacktest, queuetruth_hftbacktest not installed here (pip install hftbacktest queuetruth_hftbacktest)
the settings the line applies: {'queue_model': 'power_prob_queue_model3', 'queue_model_args': [3.0], 'exchange': 'no_partial_fill_exchange'}


## 4. How wrong was the run?

`qt.check` replays an hftbacktest run's own resting limit orders in the exact queue of the data. Here the run is
twelve orders written as order snapshots (one JSON line per order state, as `Recorder` users usually save them);
use your own.

In [8]:
log = WORK / "hbt_snapshots.jsonl"
with log.open("w") as f:
    for k, o in enumerate(ORDERS.itertuples(), start=1):
        tick = round(o.price / 0.25)
        base = {"order_id": k, "instrument": o.symbol, "side": 1 if o.side == "buy" else -1, "price_tick": tick,
                "tick_size": 0.25, "qty": o.qty, "order_type": 0, "time_in_force": 0, "maker": 1}  # fmt: skip
        new = {"leaves_qty": o.qty, "exec_price_tick": 0, "status": 1}
        end = {"leaves_qty": 0, "exec_price_tick": tick, "status": 3} if o.filled else {**new, "status": 4}
        f.write(json.dumps({**base, **new, "local_timestamp": o.placed.value, "exch_timestamp": o.placed.value + 1000}))
        f.write(
            "\n" + json.dumps({**base, **end, "local_timestamp": o.done.value + 1000, "exch_timestamp": o.done.value})
        )
        f.write("\n")
res = qt.check(log, data=DATA, horizon="1s", window="10s", out=WORK / "check")
display(res)

order,symbol,side,price,qty,entry,entry_local,backtest_filled,backtest_fill_after_s,real_filled,real_fill_after_s,queue_ahead,outcome,pnl_gap
sim:hftbacktest:backtest:1,QTBU6,sell,5008.5,1,2026-06-16T14:38:42.000+00:00,2026-06-16 14:38:42.000 UTC,True,5,True,2.497,1,both,0
sim:hftbacktest:backtest:10,QTAU6,buy,4999.5,1,2026-06-16T17:25:05.000+00:00,2026-06-16 17:25:05.000 UTC,False,,False,,12,neither,0
sim:hftbacktest:backtest:11,QTBU6,buy,5008.25,2,2026-06-16T16:12:29.000+00:00,2026-06-16 16:12:29.000 UTC,True,5,False,,0,phantom,25
sim:hftbacktest:backtest:12,QTAU6,buy,5001.25,3,2026-06-15T15:05:05.000+00:00,2026-06-15 15:05:05.000 UTC,False,,True,0.901,17,real_only,0
sim:hftbacktest:backtest:2,QTAU6,buy,4999.75,2,2026-06-16T14:38:05.000+00:00,2026-06-16 14:38:05.000 UTC,False,,False,,8,neither,0
sim:hftbacktest:backtest:3,QTBU6,sell,5010.25,3,2026-06-15T17:09:27.000+00:00,2026-06-15 17:09:27.000 UTC,True,5,False,,49,phantom,37.5
sim:hftbacktest:backtest:4,QTAU6,buy,4999.5,1,2026-06-16T17:32:15.000+00:00,2026-06-16 17:32:15.000 UTC,False,,True,0.371,1,real_only,6.25
sim:hftbacktest:backtest:5,QTAU6,sell,5002.75,2,2026-06-15T15:39:13.000+00:00,2026-06-15 15:39:13.000 UTC,True,5,False,,52,phantom,25
sim:hftbacktest:backtest:6,QTAU6,buy,5004,3,2026-06-15T19:04:05.000+00:00,2026-06-15 19:04:05.000 UTC,False,,False,,5,neither,0
sim:hftbacktest:backtest:7,QTBU6,buy,5008.75,1,2026-06-15T15:36:19.000+00:00,2026-06-15 15:36:19.000 UTC,True,5,False,,22,phantom,6.25


In [9]:
display(
    res.table()[["symbol", "side", "price", "entry_local", "backtest_filled", "real_filled", "queue_ahead", "outcome"]]
)

,symbol,side,price,entry_local,backtest_filled,real_filled,queue_ahead,outcome
0,QTBU6,sell,5008.5,2026-06-16 14:38:42.000 UTC,True,True,1,both
1,QTAU6,buy,4999.5,2026-06-16 17:25:05.000 UTC,False,False,12,neither
2,QTBU6,buy,5008.25,2026-06-16 16:12:29.000 UTC,True,False,0,phantom
3,QTAU6,buy,5001.25,2026-06-15 15:05:05.000 UTC,False,True,17,real_only
4,QTAU6,buy,4999.75,2026-06-16 14:38:05.000 UTC,False,False,8,neither
5,QTBU6,sell,5010.25,2026-06-15 17:09:27.000 UTC,True,False,49,phantom
6,QTAU6,buy,4999.5,2026-06-16 17:32:15.000 UTC,False,True,1,real_only
7,QTAU6,sell,5002.75,2026-06-15 15:39:13.000 UTC,True,False,52,phantom
8,QTAU6,buy,5004,2026-06-15 19:04:05.000 UTC,False,False,5,neither
9,QTBU6,buy,5008.75,2026-06-15 15:36:19.000 UTC,True,False,22,phantom
